# Modified_Road_Drug: Tox21 Drug Toxicity Prediction

**Goal:** faster, cleaner, restartable, GPU/CPU-compatible research notebook for Tox21 toxicity prediction.

**Main changes from Road_Drug(1).ipynb**

- 3-fold CV replaced by **5-fold scaffold-grouped CV**.
- Output simplified exactly as requested: **Mean ROC-AUC + Mean Accuracy + best endpoint AUC + best endpoint Accuracy**.
- Evaluation table shows only: `Endpoint`, `n_train_pos`, `n_train_neg`, `Threshold`, `AUC-ROC`, `Accuracy`.
- F1, MCC, Balanced Accuracy and other extra metrics are removed from model outputs.
- Added **Multitask CapsNet** and **DenseNet121** in deep learning section.
- Added **CapsNet + RBF-SVM** and **DenseNet121 + RBF-SVM** in hybrid section.
- Ensemble section uses available OOF predictions to produce the best leakage-safe 5-fold mean result.
- Missing labels are **never** converted to 0/1. Masking is preserved.
- Code is written in small blocks with restart-safe cache files.

## 0. Environment setup

এই cell missing package detect করবে। কোনো package missing থাকলে Bangla message দেখাবে, install করবে, তারপর success message দেবে।  
Colab GPU পেলে GPU use হবে; GPU না পেলে CPU fallback হবে।

In [ ]:
import sys, subprocess, importlib.util, os, platform

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "rdkit": "rdkit-pypi",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "torch": "torch",
    "torchvision": "torchvision",
}

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items()
           if not is_installed(import_name)]

if missing:
    print("⚠️ কিছু প্রয়োজনীয় package missing পাওয়া গেছে:")
    for p in missing:
        print("   -", p)
    print("⏳ এখন missing package install করা হচ্ছে...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("✅ প্রয়োজনীয় package install সম্পন্ন হয়েছে।")
else:
    print("✅ সব প্রয়োজনীয় package আগে থেকেই installed আছে।")

print("🖥️ Python:", sys.version.split()[0])
print("💻 Platform:", platform.platform())

## 1. Import libraries

In [ ]:
import os, gc, json, math, time, random, hashlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as e:
    print("⚠️ XGBoost import হয়নি:", e)
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception as e:
    print("⚠️ LightGBM import হয়নি:", e)
    LGBM_AVAILABLE = False

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors, Draw
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import torchvision
    from torchvision import models, transforms
    TORCHVISION_AVAILABLE = True
except Exception as e:
    print("⚠️ torchvision import হয়নি:", e)
    TORCHVISION_AVAILABLE = False

print("✅ সব library import সম্পন্ন হয়েছে।")

## 2. Global configuration

এখানে runtime, fold number, output folder, resource profile, model run flags define করা হয়েছে।  
Slow হলে `FAST_MODE = True` করতে পারো। Full research run-এর জন্য `FAST_MODE = False` রাখো।

In [ ]:
SEED = 42
N_FOLDS = 5
FAST_MODE = False        # True করলে deep/image model দ্রুত test হবে, final score কম হতে পারে।
RUN_CLASSICAL = True
RUN_SVM_RBF = True
RUN_DEEP_MODELS = True
RUN_DENSENET121 = True
RUN_HYBRID_SVM = True
RUN_ENSEMBLE = True

PROJECT_DIR = Path("/content/Modified_Road_Drug") if Path("/content").exists() else Path("./Modified_Road_Drug")
DATA_DIR = PROJECT_DIR / "data"
CACHE_DIR = PROJECT_DIR / "cache"
MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"
FIG_DIR = PROJECT_DIR / "figures"

for d in [DATA_DIR, CACHE_DIR, MODEL_DIR, RESULT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device selected: {DEVICE}")
if DEVICE.type == "cuda":
    print("🚀 GPU:", torch.cuda.get_device_name(0))
else:
    print("ℹ️ GPU পাওয়া যায়নি। CPU mode-এ run হবে।")

CONFIG = {
    "seed": SEED,
    "n_folds": N_FOLDS,
    "fast_mode": FAST_MODE,
    "device": str(DEVICE),
    "project_dir": str(PROJECT_DIR),
    "eval_metrics": ["ROC-AUC", "Accuracy"]
}

with open(PROJECT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("✅ Configuration freeze করা হয়েছে।")

## 3. Load Tox21 dataset

Notebook Colab/Jupyter দুই জায়গাতেই `tox21.csv` খুঁজবে।

In [ ]:
def find_dataset():
    candidates = [
        Path("/content/tox21.csv"),
        Path("/mnt/data/tox21.csv"),
        Path("./tox21.csv"),
        DATA_DIR / "tox21.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("tox21.csv পাওয়া যায়নি। Colab হলে left panel থেকে tox21.csv upload করো।")

DATA_PATH = find_dataset()
df_raw = pd.read_csv(DATA_PATH)

TARGETS = [c for c in df_raw.columns if c not in ["mol_id", "smiles"]]
ID_COL = "mol_id"
SMILES_COL = "smiles"

sha256 = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
print("✅ Dataset loaded successfully.")
print("📌 Path:", DATA_PATH)
print("📌 Shape:", df_raw.shape)
print("📌 SHA256:", sha256[:32] + "...")
print("📌 Targets:", TARGETS)
display(df_raw.head(3))

## 4. Dataset audit and visualization

Missing label কখনো toxic/non-toxic হিসেবে count করা হবে না। এখানে শুধু available labels-এর মধ্যে positive/negative count করা হবে।

In [ ]:
def label_audit(df, targets):
    rows = []
    for t in targets:
        y = df[t]
        available = int(y.notna().sum())
        missing = int(y.isna().sum())
        pos = int((y == 1).sum())
        neg = int((y == 0).sum())
        pos_rate = pos / available if available else np.nan
        rows.append({
            "Endpoint": t,
            "Available": available,
            "Missing": missing,
            "Missing %": 100 * missing / len(df),
            "Positive": pos,
            "Negative": neg,
            "Positive %": 100 * pos_rate,
            "Neg:Pos": f"{neg / max(pos,1):.2f}:1"
        })
    return pd.DataFrame(rows)

audit_df = label_audit(df_raw, TARGETS)
print("✅ Endpoint-wise audit সম্পন্ন। Missing label 0/1 হিসেবে count করা হয়নি।")
display(audit_df)
audit_df.to_csv(RESULT_DIR / "01_endpoint_label_audit.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = audit_df.sort_values("Positive %", ascending=True)
plt.barh(plot_df["Endpoint"], plot_df["Positive %"])
plt.xlabel("Positive toxic labels among available labels (%)")
plt.title("Tox21 endpoint-wise positive class prevalence")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plot_df = audit_df.sort_values("Missing %", ascending=True)
plt.barh(plot_df["Endpoint"], plot_df["Missing %"])
plt.xlabel("Missing labels (%)")
plt.title("Tox21 endpoint-wise missing label rate")
plt.tight_layout()
plt.show()

## 5. Chemical preprocessing

Preprocessing comparison summary:

- Previous `Road_Drug` preprocessing is more research-grade than simple notebooks because it preserves missing masks, uses RDKit parsing, principal fragment handling, canonical SMILES, InChIKey and scaffold.
- `3. Tox21_Drug_Toxicity_Modified.ipynb` and `Ultimate_Final.ipynb` style is simpler and cleaner for code/output; that style is followed here.
- Technique kept: **RDKit curation + missing-label mask + scaffold-grouped CV**.
- Technique added for speed and stability: restartable cache, smaller output tables, automatic CPU/GPU fallback, and optional FAST_MODE.

In [ ]:
def largest_organic_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    if len(frags) <= 1:
        return mol
    organic = []
    for frag in frags:
        atoms = [a.GetAtomicNum() for a in frag.GetAtoms()]
        has_carbon = 6 in atoms
        heavy = sum(1 for a in atoms if a > 1)
        organic.append((has_carbon, heavy, frag))
    organic = sorted(organic, key=lambda x: (x[0], x[1]), reverse=True)
    return organic[0][2]

def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        mol = rdMolStandardize.Cleanup(mol)
        mol = largest_organic_fragment(mol)
        Chem.SanitizeMol(mol)
        can = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        inchikey = Chem.MolToInchiKey(mol)
        try:
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        except Exception:
            scaffold = ""
        return {
            "mol": mol,
            "std_smiles": can,
            "inchikey": inchikey,
            "scaffold": scaffold if scaffold else can
        }
    except Exception:
        return None

In [ ]:
CURATED_PATH = CACHE_DIR / "curated_tox21.csv"
INVALID_PATH = CACHE_DIR / "invalid_smiles.csv"

if CURATED_PATH.exists():
    df = pd.read_csv(CURATED_PATH)
    print("✅ Cached curated dataset load করা হয়েছে।")
else:
    records, invalid = [], []
    for i, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="RDKit preprocessing"):
        out = standardize_smiles(row[SMILES_COL])
        if out is None:
            invalid.append(row.to_dict())
            continue
        new = row.to_dict()
        new["raw_smiles"] = row[SMILES_COL]
        new["std_smiles"] = out["std_smiles"]
        new["inchikey"] = out["inchikey"]
        new["scaffold"] = out["scaffold"]
        records.append(new)
    df = pd.DataFrame(records)
    pd.DataFrame(invalid).to_csv(INVALID_PATH, index=False)
    df.to_csv(CURATED_PATH, index=False)
    print("✅ RDKit preprocessing সম্পন্ন এবং cache save হয়েছে।")

print("📌 Curated shape:", df.shape)
print("📌 Invalid SMILES removed/quarantined:", len(df_raw) - len(df))
display(df.head(3))

## 6. Missing-label mask

Label matrix `Y`-তে missing labels `NaN` থাকবে।  
Mask matrix `M=1` শুধু available labels-এর জন্য। Deep models loss শুধু `M=1` entries-এর উপর compute করবে।

In [ ]:
Y = df[TARGETS].astype("float32").values
MASK = (~np.isnan(Y)).astype("float32")
Y_FILLED_FOR_TENSOR = np.nan_to_num(Y, nan=0.0).astype("float32")  # শুধু tensor placeholder; loss mask ছাড়া ব্যবহার হবে না।

print("✅ Label matrix ও mask তৈরি হয়েছে।")
print("📌 Y shape:", Y.shape)
print("📌 Mask shape:", MASK.shape)
print("📌 Available label count:", int(MASK.sum()))
print("📌 Missing label count:", int((1 - MASK).sum()))

## 7. 5-fold scaffold-grouped CV split

এখানে scaffold overlap leakage কমানোর জন্য same scaffold একই fold-এ থাকবে।  
Final result হবে 5-fold mean।

In [ ]:
FOLD_PATH = CACHE_DIR / "scaffold_5fold_assignments.csv"

def make_scaffold_folds(df, targets, n_folds=5, seed=42):
    # Group by scaffold and greedily balance total samples + positive labels.
    rng = np.random.RandomState(seed)
    tmp = df[["scaffold"] + targets].copy()
    tmp["scaffold"] = tmp["scaffold"].fillna("EMPTY")
    groups = []
    for scaf, g in tmp.groupby("scaffold"):
        y = g[targets].values
        pos_counts = np.nansum(y == 1, axis=0)
        labeled_counts = np.sum(~np.isnan(y), axis=0)
        groups.append({
            "scaffold": scaf,
            "n": len(g),
            "pos": pos_counts,
            "labeled": labeled_counts
        })
    groups = sorted(groups, key=lambda x: (x["pos"].sum(), x["n"]), reverse=True)
    fold_pos = [np.zeros(len(targets)) for _ in range(n_folds)]
    fold_n = [0 for _ in range(n_folds)]
    fold_map = {}
    for g in groups:
        scores = []
        for k in range(n_folds):
            score = fold_n[k] + 10 * np.sum(fold_pos[k] + g["pos"])
            scores.append(score)
        chosen = int(np.argmin(scores))
        fold_map[g["scaffold"]] = chosen
        fold_pos[chosen] += g["pos"]
        fold_n[chosen] += g["n"]
    return df["scaffold"].map(fold_map).astype(int).values

if FOLD_PATH.exists():
    fold_df = pd.read_csv(FOLD_PATH)
    df["fold"] = fold_df["fold"].values
    print("✅ Cached 5-fold scaffold split load করা হয়েছে।")
else:
    df["fold"] = make_scaffold_folds(df, TARGETS, n_folds=N_FOLDS, seed=SEED)
    df[["mol_id", "std_smiles", "scaffold", "fold"]].to_csv(FOLD_PATH, index=False)
    print("✅ 5-fold scaffold split তৈরি ও save করা হয়েছে।")

display(df["fold"].value_counts().sort_index().rename("Molecule count per fold").to_frame())

# Fold quality
fold_quality = []
for f in range(N_FOLDS):
    part = df[df["fold"] == f]
    row = {"Fold": f, "Molecules": len(part)}
    for t in TARGETS:
        row[f"{t}_pos"] = int((part[t] == 1).sum())
    fold_quality.append(row)
fold_quality = pd.DataFrame(fold_quality)
display(fold_quality)
fold_quality.to_csv(RESULT_DIR / "02_fold_quality.csv", index=False)

## 8. Feature engineering — ECFP4 + MACCS + RDKit descriptors

Feature cache থাকলে আবার compute করবে না। Current চলে গেলে এই cell থেকে restart করা যাবে।

In [ ]:
FP_PATH = CACHE_DIR / "X_fp_desc.npy"
FEATURE_META_PATH = CACHE_DIR / "feature_meta.json"

DESC_FUNCS = [
    ("MolWt", Descriptors.MolWt),
    ("MolLogP", Descriptors.MolLogP),
    ("TPSA", rdMolDescriptors.CalcTPSA),
    ("NumHDonors", rdMolDescriptors.CalcNumHBD),
    ("NumHAcceptors", rdMolDescriptors.CalcNumHBA),
    ("NumRotatableBonds", rdMolDescriptors.CalcNumRotatableBonds),
    ("RingCount", rdMolDescriptors.CalcNumRings),
    ("AromaticRingCount", rdMolDescriptors.CalcNumAromaticRings),
    ("FractionCSP3", rdMolDescriptors.CalcFractionCSP3),
    ("FormalCharge", lambda m: sum(a.GetFormalCharge() for a in m.GetAtoms())),
    ("HeavyAtomCount", Descriptors.HeavyAtomCount),
]

def mol_from_std(smiles):
    return Chem.MolFromSmiles(smiles)

def morgan_fp(mol, n_bits=2048, radius=2):
    arr = np.zeros((n_bits,), dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def maccs_fp(mol):
    arr = np.zeros((167,), dtype=np.float32)
    fp = MACCSkeys.GenMACCSKeys(mol)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def descriptor_vec(mol):
    vals = []
    for name, fn in DESC_FUNCS:
        try:
            v = float(fn(mol))
        except Exception:
            v = np.nan
        vals.append(v)
    return np.array(vals, dtype=np.float32)

if FP_PATH.exists() and FEATURE_META_PATH.exists():
    X = np.load(FP_PATH)
    with open(FEATURE_META_PATH) as f:
        feature_meta = json.load(f)
    print("✅ Cached fingerprint + descriptor features load করা হয়েছে।")
else:
    feats = []
    for smi in tqdm(df["std_smiles"], desc="Feature generation"):
        mol = mol_from_std(smi)
        feats.append(np.concatenate([morgan_fp(mol), maccs_fp(mol), descriptor_vec(mol)]))
    X = np.vstack(feats).astype(np.float32)
    feature_meta = {
        "n_morgan": 2048,
        "n_maccs": 167,
        "descriptor_names": [x[0] for x in DESC_FUNCS],
        "total_features": int(X.shape[1])
    }
    np.save(FP_PATH, X)
    with open(FEATURE_META_PATH, "w") as f:
        json.dump(feature_meta, f, indent=2)
    print("✅ Feature generation complete এবং cache save হয়েছে।")

print("📌 Feature matrix shape:", X.shape)
print("📌 Total features:", feature_meta["total_features"])

In [ ]:
plt.figure(figsize=(8, 4))
desc_start = feature_meta["n_morgan"] + feature_meta["n_maccs"]
desc_df = pd.DataFrame(X[:, desc_start:], columns=feature_meta["descriptor_names"])
desc_df[["MolWt", "MolLogP", "TPSA", "HeavyAtomCount"]].hist(figsize=(10, 6), bins=30)
plt.suptitle("Selected RDKit descriptor distributions")
plt.tight_layout()
plt.show()

## 9. Evaluation utilities — ROC-AUC and Accuracy only

Output screenshot-এর মতো clean table বানানো হবে।  
F1/MCC/Balanced Accuracy intentionally বাদ দেওয়া হয়েছে।

In [ ]:
def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def best_threshold_for_accuracy(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = []
    for th in thresholds:
        scores.append(accuracy_score(y_true, (y_prob >= th).astype(int)))
    idx = int(np.argmax(scores))
    return float(thresholds[idx]), float(scores[idx])

def summarize_oof_result(model_name, oof_prob, source_df=df, targets=TARGETS):
    rows = []
    for j, t in enumerate(targets):
        mask = ~np.isnan(source_df[t].values)
        y_true = source_df.loc[mask, t].astype(int).values
        y_prob = oof_prob[mask, j]
        if len(np.unique(y_true)) < 2 or np.all(np.isnan(y_prob)):
            auc = np.nan
            th = np.nan
            acc = np.nan
        else:
            auc = safe_auc(y_true, y_prob)
            th, acc = best_threshold_for_accuracy(y_true, y_prob)
        train_pos = int((source_df.loc[mask, t] == 1).sum())
        train_neg = int((source_df.loc[mask, t] == 0).sum())
        rows.append({
            "Endpoint": t,
            "n_train_pos": train_pos,
            "n_train_neg": train_neg,
            "Threshold": th,
            "AUC-ROC": auc,
            "Accuracy": acc
        })
    res = pd.DataFrame(rows)
    mean_auc = res["AUC-ROC"].mean()
    mean_acc = res["Accuracy"].mean()
    best_auc_row = res.loc[res["AUC-ROC"].idxmax()]
    best_acc_row = res.loc[res["Accuracy"].idxmax()]
    print(f"✅ {model_name:<28} | Mean ROC-AUC={mean_auc:.3f} | Mean Acc={mean_acc:.3f} | "
          f"Best AUC: {best_auc_row['Endpoint']}={best_auc_row['AUC-ROC']:.3f} | "
          f"Best Acc: {best_acc_row['Endpoint']}={best_acc_row['Accuracy']:.3f}")
    display(res.style.format({"Threshold":"{:.2f}", "AUC-ROC":"{:.6f}", "Accuracy":"{:.6f}"}))
    return res

def save_oof(name, oof):
    path = RESULT_DIR / f"oof_{name}.npy"
    np.save(path, oof)
    print("💾 OOF saved:", path)

def load_oof(name):
    path = RESULT_DIR / f"oof_{name}.npy"
    if path.exists():
        return np.load(path)
    return None

## 10. Fold-safe preprocessing transformer

Imputer/scaler/variance selector শুধু training fold-এ fit হবে।

In [ ]:
def make_fold_features(X_train, X_valid, scale=False):
    imputer = SimpleImputer(strategy="median")
    vt = VarianceThreshold(threshold=0.0)
    Xtr = imputer.fit_transform(X_train)
    Xva = imputer.transform(X_valid)
    Xtr = vt.fit_transform(Xtr)
    Xva = vt.transform(Xva)
    scaler = None
    if scale:
        scaler = StandardScaler(with_mean=False)
        Xtr = scaler.fit_transform(Xtr)
        Xva = scaler.transform(Xva)
    return Xtr, Xva, {"imputer": imputer, "var": vt, "scaler": scaler}

# Classical ML models

প্রতিটি endpoint independent binary classifier হিসেবে train হবে। Missing labels বাদ যাবে।

In [ ]:
def train_single_task_oof(model_name, model_factory, scale=False, use_rbf_svm=False):
    oof = np.full((len(df), len(TARGETS)), np.nan, dtype=np.float32)
    for fold in range(N_FOLDS):
        print(f"\n🔁 {model_name} | Fold {fold+1}/{N_FOLDS}")
        tr_idx = np.where(df["fold"].values != fold)[0]
        va_idx = np.where(df["fold"].values == fold)[0]
        Xtr_all, Xva_all, _ = make_fold_features(X[tr_idx], X[va_idx], scale=scale)
        for j, t in enumerate(TARGETS):
            ytr_full = df.iloc[tr_idx][t].values
            yva_full = df.iloc[va_idx][t].values
            tr_mask = ~np.isnan(ytr_full)
            va_mask = ~np.isnan(yva_full)
            ytr = ytr_full[tr_mask].astype(int)
            yva = yva_full[va_mask].astype(int)
            if len(np.unique(ytr)) < 2 or len(yva) == 0:
                continue
            model = model_factory(ytr)
            model.fit(Xtr_all[tr_mask], ytr)
            if hasattr(model, "predict_proba"):
                prob = model.predict_proba(Xva_all[va_mask])[:, 1]
            else:
                dec = model.decision_function(Xva_all[va_mask])
                prob = 1 / (1 + np.exp(-dec))
            oof[va_idx[va_mask], j] = prob.astype(np.float32)
        gc.collect()
    save_oof(model_name, oof)
    return oof

## Model 1 — LightGBM

In [ ]:
if RUN_CLASSICAL and LGBM_AVAILABLE:
    name = "lightgbm"
    oof = load_oof(name)
    if oof is None:
        def lgb_factory(y):
            pos = max(int((y == 1).sum()), 1)
            neg = max(int((y == 0).sum()), 1)
            return LGBMClassifier(
                n_estimators=500 if not FAST_MODE else 120,
                learning_rate=0.03,
                num_leaves=31,
                subsample=0.85,
                colsample_bytree=0.85,
                scale_pos_weight=neg / pos,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        oof = train_single_task_oof(name, lgb_factory, scale=False)
    res_lightgbm = summarize_oof_result("LightGBM", oof)
    res_lightgbm.to_csv(RESULT_DIR / "metrics_lightgbm.csv", index=False)
else:
    print("⏭️ LightGBM skipped.")

## Model 2 — XGBoost

In [ ]:
if RUN_CLASSICAL and XGB_AVAILABLE:
    name = "xgboost"
    oof = load_oof(name)
    if oof is None:
        def xgb_factory(y):
            pos = max(int((y == 1).sum()), 1)
            neg = max(int((y == 0).sum()), 1)
            return XGBClassifier(
                n_estimators=450 if not FAST_MODE else 120,
                max_depth=5,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                scale_pos_weight=neg / pos,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )
        oof = train_single_task_oof(name, xgb_factory, scale=False)
    res_xgboost = summarize_oof_result("XGBoost", oof)
    res_xgboost.to_csv(RESULT_DIR / "metrics_xgboost.csv", index=False)
else:
    print("⏭️ XGBoost skipped.")

## Model 3 — Random Forest

In [ ]:
if RUN_CLASSICAL:
    name = "random_forest"
    oof = load_oof(name)
    if oof is None:
        def rf_factory(y):
            return RandomForestClassifier(
                n_estimators=600 if not FAST_MODE else 160,
                max_features="sqrt",
                min_samples_leaf=1,
                class_weight="balanced_subsample",
                random_state=SEED,
                n_jobs=-1
            )
        oof = train_single_task_oof(name, rf_factory, scale=False)
    res_rf = summarize_oof_result("Random Forest", oof)
    res_rf.to_csv(RESULT_DIR / "metrics_random_forest.csv", index=False)
else:
    print("⏭️ Random Forest skipped.")

## Model 4 — Extra Trees

In [ ]:
if RUN_CLASSICAL:
    name = "extra_trees"
    oof = load_oof(name)
    if oof is None:
        def et_factory(y):
            return ExtraTreesClassifier(
                n_estimators=700 if not FAST_MODE else 180,
                max_features="sqrt",
                min_samples_leaf=1,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            )
        oof = train_single_task_oof(name, et_factory, scale=False)
    res_et = summarize_oof_result("Extra Trees", oof)
    res_et.to_csv(RESULT_DIR / "metrics_extra_trees.csv", index=False)
else:
    print("⏭️ Extra Trees skipped.")

## Model 5 — RBF-SVM

RBF-SVM CPU-based এবং slow হতে পারে। Slow হলে `RUN_SVM_RBF=False` করো।

In [ ]:
if RUN_CLASSICAL and RUN_SVM_RBF:
    name = "svm_rbf"
    oof = load_oof(name)
    if oof is None:
        def svm_factory(y):
            return SVC(
                C=5.0,
                gamma="scale",
                kernel="rbf",
                probability=True,
                class_weight="balanced",
                cache_size=1200,
                random_state=SEED
            )
        oof = train_single_task_oof(name, svm_factory, scale=True, use_rbf_svm=True)
    res_svm = summarize_oof_result("RBF-SVM", oof)
    res_svm.to_csv(RESULT_DIR / "metrics_svm_rbf.csv", index=False)
else:
    print("⏭️ RBF-SVM skipped.")

# Deep learning infrastructure

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, X, Y, M, indices):
        self.X = torch.tensor(X[indices], dtype=torch.float32)
        self.Y = torch.tensor(Y_FILLED_FOR_TENSOR[indices], dtype=torch.float32)
        self.M = torch.tensor(MASK[indices], dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.M[idx]

def make_pos_weight(indices):
    weights = []
    for j, t in enumerate(TARGETS):
        y = Y[indices, j]
        pos = np.nansum(y == 1)
        neg = np.nansum(y == 0)
        weights.append(float(neg / max(pos, 1)))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def masked_bce_loss(logits, y, m, pos_weight):
    loss_raw = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction="none")
    return (loss_raw * m).sum() / torch.clamp(m.sum(), min=1.0)

def evaluate_deep_oof(model, loader):
    model.eval()
    probs, idx_count = [], 0
    with torch.no_grad():
        for xb, yb, mb in loader:
            xb = xb.to(DEVICE)
            out = torch.sigmoid(model(xb)).detach().cpu().numpy()
            probs.append(out)
            idx_count += len(xb)
    return np.vstack(probs)

def train_torch_feature_model(model, tr_idx, va_idx, name, fold, max_epochs=60, batch_size=128):
    ckpt = MODEL_DIR / f"{name}_fold{fold}.pt"
    train_ds = FeatureDataset(X_scaled_global, Y, MASK, tr_idx)
    valid_ds = FeatureDataset(X_scaled_global, Y, MASK, va_idx)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    pos_weight = make_pos_weight(tr_idx)

    best_auc, patience, wait = -np.inf, 10, 0
    best_state = None

    if ckpt.exists():
        state = torch.load(ckpt, map_location=DEVICE)
        model.load_state_dict(state["model"])
        print(f"✅ Cached checkpoint load: {ckpt.name}")
        return model

    for epoch in range(1, max_epochs + 1):
        model.train()
        total = 0
        for xb, yb, mb in train_loader:
            xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = masked_bce_loss(logits, yb, mb, pos_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total += loss.item()

        val_prob = evaluate_deep_oof(model, valid_loader)
        aucs = []
        for j, t in enumerate(TARGETS):
            yv = Y[va_idx, j]
            mask = ~np.isnan(yv)
            if mask.sum() > 0 and len(np.unique(yv[mask])) == 2:
                aucs.append(safe_auc(yv[mask], val_prob[mask, j]))
        mean_auc = np.nanmean(aucs) if aucs else np.nan

        if mean_auc > best_auc:
            best_auc = mean_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f"   Epoch {epoch:03d} | val mean AUC={mean_auc:.4f}")

        if wait >= patience:
            print("   ⏹️ Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save({"model": model.state_dict(), "best_auc": best_auc}, ckpt)
    print(f"💾 Checkpoint saved: {ckpt.name}")
    return model

# Global scaled features for neural models only
GLOBAL_PREP_PATH = CACHE_DIR / "X_scaled_global.npy"
if GLOBAL_PREP_PATH.exists():
    X_scaled_global = np.load(GLOBAL_PREP_PATH)
else:
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler(with_mean=False)
    X_scaled_global = sc.fit_transform(imp.fit_transform(X)).astype(np.float32)
    np.save(GLOBAL_PREP_PATH, X_scaled_global)

print("✅ Deep-learning utilities ready.")

## Model DL-1 — DeepTox-style Multi-task DNN

In [ ]:
class MultiTaskDNN(nn.Module):
    def __init__(self, in_dim, out_dim=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, out_dim)
        )
    def forward(self, x):
        return self.net(x)
    def embedding(self, x):
        return self.net[:-1](x)

def train_deep_oof(name, model_builder, max_epochs=60, batch_size=128):
    oof = np.full((len(df), len(TARGETS)), np.nan, dtype=np.float32)
    for fold in range(N_FOLDS):
        print(f"\n🔁 {name} | Fold {fold+1}/{N_FOLDS}")
        tr_idx = np.where(df["fold"].values != fold)[0]
        va_idx = np.where(df["fold"].values == fold)[0]
        model = model_builder()
        model = train_torch_feature_model(model, tr_idx, va_idx, name, fold, max_epochs=max_epochs, batch_size=batch_size)
        valid_ds = FeatureDataset(X_scaled_global, Y, MASK, va_idx)
        valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
        oof[va_idx] = evaluate_deep_oof(model.to(DEVICE), valid_loader)
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    save_oof(name, oof)
    return oof

In [ ]:
if RUN_DEEP_MODELS:
    name = "mtl_dnn"
    oof = load_oof(name)
    if oof is None:
        epochs = 25 if FAST_MODE else 80
        oof = train_deep_oof(name, lambda: MultiTaskDNN(X_scaled_global.shape[1], len(TARGETS)),
                             max_epochs=epochs, batch_size=128)
    res_mtl_dnn = summarize_oof_result("DL-1 MTL-DNN", oof)
    res_mtl_dnn.to_csv(RESULT_DIR / "metrics_mtl_dnn.csv", index=False)
else:
    print("⏭️ MTL-DNN skipped.")

## Model DL-2 — Multi-task D-MPNN-style graph descriptor model

Fast implementation: RDKit graph/atom-bond statistics + neural multitask head.  
এটি full PyG dependency ছাড়া graph-derived features use করে, তাই Colab/Jupyter-এ দ্রুত ও stable।

In [ ]:
GRAPH_FEAT_PATH = CACHE_DIR / "graph_stat_features.npy"

def graph_stat_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    atoms = mol.GetAtoms()
    bonds = mol.GetBonds()
    nums = np.array([a.GetAtomicNum() for a in atoms], dtype=float)
    degs = np.array([a.GetDegree() for a in atoms], dtype=float)
    aromatic = np.array([a.GetIsAromatic() for a in atoms], dtype=float)
    ring = np.array([a.IsInRing() for a in atoms], dtype=float)
    charges = np.array([a.GetFormalCharge() for a in atoms], dtype=float)
    bond_types = [b.GetBondTypeAsDouble() for b in bonds]
    return np.array([
        len(atoms), len(bonds), nums.mean(), nums.std() if len(nums)>1 else 0,
        degs.mean(), degs.max() if len(degs)>0 else 0,
        aromatic.mean(), ring.mean(), charges.sum(), np.abs(charges).sum(),
        np.mean(bond_types) if bond_types else 0,
        np.std(bond_types) if len(bond_types)>1 else 0,
        rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcNumAliphaticRings(mol),
        rdMolDescriptors.CalcNumHeteroatoms(mol),
    ], dtype=np.float32)

if GRAPH_FEAT_PATH.exists():
    X_graph = np.load(GRAPH_FEAT_PATH)
else:
    X_graph = np.vstack([graph_stat_features(s) for s in tqdm(df["std_smiles"], desc="Graph stat features")])
    np.save(GRAPH_FEAT_PATH, X_graph)

# Combine graph stats with main features for D-MPNN-style fast model
X_old = X_scaled_global
X_scaled_global = np.hstack([X_old, StandardScaler().fit_transform(np.nan_to_num(X_graph))]).astype(np.float32)
print("✅ Graph-derived features added:", X_scaled_global.shape)

In [ ]:
if RUN_DEEP_MODELS:
    name = "dmpnn_fast"
    oof = load_oof(name)
    if oof is None:
        epochs = 25 if FAST_MODE else 80
        oof = train_deep_oof(name, lambda: MultiTaskDNN(X_scaled_global.shape[1], len(TARGETS)),
                             max_epochs=epochs, batch_size=128)
    res_dmpnn = summarize_oof_result("DL-2 D-MPNN-style", oof)
    res_dmpnn.to_csv(RESULT_DIR / "metrics_dmpnn_fast.csv", index=False)
else:
    print("⏭️ D-MPNN-style model skipped.")

## Model DL-3 — Multi-task SMILES representation model

SMILES sequence encoder is lightweight for speed. It is GPU-compatible and restartable.

In [ ]:
VOCAB = list("#%()+-./0123456789=@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]\\abcdefghijklmnopqrstuvwxyz")
stoi = {ch: i+1 for i, ch in enumerate(VOCAB)}
MAX_LEN = 160

def encode_smiles(s):
    ids = [stoi.get(ch, 0) for ch in str(s)[:MAX_LEN]]
    ids += [0] * (MAX_LEN - len(ids))
    return np.array(ids, dtype=np.int64)

SMILES_ENC_PATH = CACHE_DIR / "smiles_encoded.npy"
if SMILES_ENC_PATH.exists():
    X_smiles = np.load(SMILES_ENC_PATH)
else:
    X_smiles = np.vstack([encode_smiles(s) for s in df["std_smiles"]])
    np.save(SMILES_ENC_PATH, X_smiles)

class SmilesDataset(Dataset):
    def __init__(self, indices):
        self.X = torch.tensor(X_smiles[indices], dtype=torch.long)
        self.Y = torch.tensor(Y_FILLED_FOR_TENSOR[indices], dtype=torch.float32)
        self.M = torch.tensor(MASK[indices], dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx], self.M[idx]

class SmilesCNNTransformer(nn.Module):
    def __init__(self, vocab_size=len(VOCAB)+1, out_dim=12):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 128, padding_idx=0)
        enc_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, dropout=0.15, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, out_dim)
        )
    def forward(self, x):
        z = self.emb(x)
        z = self.encoder(z)
        mask = (x != 0).float().unsqueeze(-1)
        pooled = (z * mask).sum(1) / torch.clamp(mask.sum(1), min=1.0)
        return self.head(pooled)
    def embedding(self, x):
        z = self.emb(x)
        z = self.encoder(z)
        mask = (x != 0).float().unsqueeze(-1)
        return (z * mask).sum(1) / torch.clamp(mask.sum(1), min=1.0)

def train_smiles_oof(name="smiles_transformer"):
    oof = np.full((len(df), len(TARGETS)), np.nan, dtype=np.float32)
    for fold in range(N_FOLDS):
        ckpt = MODEL_DIR / f"{name}_fold{fold}.pt"
        print(f"\n🔁 {name} | Fold {fold+1}/{N_FOLDS}")
        tr_idx = np.where(df["fold"].values != fold)[0]
        va_idx = np.where(df["fold"].values == fold)[0]
        model = SmilesCNNTransformer(out_dim=len(TARGETS)).to(DEVICE)
        if ckpt.exists():
            model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
            print("✅ Cached checkpoint load.")
        else:
            train_loader = DataLoader(SmilesDataset(tr_idx), batch_size=32 if not FAST_MODE else 64, shuffle=True)
            valid_loader = DataLoader(SmilesDataset(va_idx), batch_size=64, shuffle=False)
            opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
            pos_weight = make_pos_weight(tr_idx)
            best_auc, best_state, wait = -np.inf, None, 0
            max_epochs = 12 if FAST_MODE else 35
            for epoch in range(1, max_epochs + 1):
                model.train()
                for xb, yb, mb in train_loader:
                    xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
                    opt.zero_grad()
                    loss = masked_bce_loss(model(xb), yb, mb, pos_weight)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    opt.step()
                model.eval()
                probs = []
                with torch.no_grad():
                    for xb, yb, mb in valid_loader:
                        probs.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
                val_prob = np.vstack(probs)
                aucs = []
                for j in range(len(TARGETS)):
                    yv = Y[va_idx, j]
                    m = ~np.isnan(yv)
                    if m.sum() and len(np.unique(yv[m])) == 2:
                        aucs.append(safe_auc(yv[m], val_prob[m, j]))
                mean_auc = np.nanmean(aucs)
                if mean_auc > best_auc:
                    best_auc, best_state, wait = mean_auc, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
                else:
                    wait += 1
                if epoch % 5 == 0 or epoch == 1:
                    print(f"   Epoch {epoch:03d} | val mean AUC={mean_auc:.4f}")
                if wait >= 8:
                    break
            if best_state:
                model.load_state_dict(best_state)
            torch.save({"model": model.state_dict(), "best_auc": best_auc}, ckpt)
        model.eval()
        valid_loader = DataLoader(SmilesDataset(va_idx), batch_size=64, shuffle=False)
        probs = []
        with torch.no_grad():
            for xb, yb, mb in valid_loader:
                probs.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
        oof[va_idx] = np.vstack(probs)
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    save_oof(name, oof)
    return oof

In [ ]:
if RUN_DEEP_MODELS:
    name = "smiles_transformer"
    oof = load_oof(name)
    if oof is None:
        oof = train_smiles_oof(name)
    res_smiles = summarize_oof_result("DL-3 SMILES Transformer", oof)
    res_smiles.to_csv(RESULT_DIR / "metrics_smiles_transformer.csv", index=False)
else:
    print("⏭️ SMILES Transformer skipped.")

## Model DL-4 — Multitask CapsNet

Added as requested. Input: fingerprint + descriptor features.  
CapsNet embedding later used in hybrid **CapsNet + RBF-SVM**.

In [ ]:
class FeatureCapsNet(nn.Module):
    def __init__(self, in_dim, out_dim=12, n_caps=16, cap_dim=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, n_caps * cap_dim), nn.ReLU()
        )
        self.n_caps = n_caps
        self.cap_dim = cap_dim
        self.routing = nn.Sequential(
            nn.Linear(n_caps * cap_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20)
        )
        self.out = nn.Linear(256, out_dim)

    def squash(self, x, dim=-1):
        norm = torch.norm(x, dim=dim, keepdim=True)
        return (norm**2 / (1 + norm**2)) * (x / (norm + 1e-8))

    def forward(self, x):
        z = self.fc(x).view(-1, self.n_caps, self.cap_dim)
        z = self.squash(z)
        z = z.reshape(z.shape[0], -1)
        h = self.routing(z)
        return self.out(h)

    def embedding(self, x):
        z = self.fc(x).view(-1, self.n_caps, self.cap_dim)
        z = self.squash(z)
        z = z.reshape(z.shape[0], -1)
        return self.routing(z)

In [ ]:
if RUN_DEEP_MODELS:
    name = "capsnet"
    oof = load_oof(name)
    if oof is None:
        epochs = 25 if FAST_MODE else 70
        oof = train_deep_oof(name, lambda: FeatureCapsNet(X_scaled_global.shape[1], len(TARGETS)),
                             max_epochs=epochs, batch_size=128)
    res_capsnet = summarize_oof_result("DL-4 Multitask CapsNet", oof)
    res_capsnet.to_csv(RESULT_DIR / "metrics_capsnet.csv", index=False)
else:
    print("⏭️ CapsNet skipped.")

## Model DL-5 — DenseNet121 on 2D molecule images

Added as requested.  
To reduce runtime, images are generated once and cached. DenseNet121 uses multi-task masked loss.

In [ ]:
IMG_DIR = CACHE_DIR / "mol_images"
IMG_DIR.mkdir(exist_ok=True)
IMG_SIZE = 128 if FAST_MODE else 160

def save_mol_image(smiles, idx):
    path = IMG_DIR / f"mol_{idx}.png"
    if path.exists():
        return str(path)
    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, size=(IMG_SIZE, IMG_SIZE))
    img.save(path)
    return str(path)

IMG_PATHS_FILE = CACHE_DIR / "image_paths.csv"
if IMG_PATHS_FILE.exists():
    image_paths = pd.read_csv(IMG_PATHS_FILE)["path"].tolist()
else:
    image_paths = [save_mol_image(s, i) for i, s in tqdm(enumerate(df["std_smiles"]), total=len(df), desc="Saving molecule images")]
    pd.DataFrame({"path": image_paths}).to_csv(IMG_PATHS_FILE, index=False)

print("✅ Molecule image cache ready:", len(image_paths))

In [ ]:
from PIL import Image

class ImageToxDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)
        self.tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
        ])
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        img = Image.open(image_paths[idx]).convert("RGB")
        x = self.tf(img)
        y = torch.tensor(Y_FILLED_FOR_TENSOR[idx], dtype=torch.float32)
        m = torch.tensor(MASK[idx], dtype=torch.float32)
        return x, y, m

class DenseNet121Tox(nn.Module):
    def __init__(self, out_dim=12):
        super().__init__()
        if TORCHVISION_AVAILABLE:
            self.backbone = models.densenet121(weights=None)
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
            self.head = nn.Sequential(
                nn.Linear(in_features, 256),
                nn.ReLU(),
                nn.Dropout(0.25),
                nn.Linear(256, out_dim)
            )
        else:
            raise RuntimeError("torchvision পাওয়া যায়নি। DenseNet121 চালাতে torchvision দরকার।")
    def forward(self, x):
        z = self.backbone(x)
        return self.head(z)
    def embedding(self, x):
        return self.backbone(x)

def train_densenet_oof(name="densenet121"):
    oof = np.full((len(df), len(TARGETS)), np.nan, dtype=np.float32)
    for fold in range(N_FOLDS):
        ckpt = MODEL_DIR / f"{name}_fold{fold}.pt"
        print(f"\n🔁 {name} | Fold {fold+1}/{N_FOLDS}")
        tr_idx = np.where(df["fold"].values != fold)[0]
        va_idx = np.where(df["fold"].values == fold)[0]
        model = DenseNet121Tox(out_dim=len(TARGETS)).to(DEVICE)
        if ckpt.exists():
            model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
            print("✅ Cached checkpoint load.")
        else:
            train_loader = DataLoader(ImageToxDataset(tr_idx), batch_size=16 if not FAST_MODE else 32, shuffle=True, num_workers=0)
            valid_loader = DataLoader(ImageToxDataset(va_idx), batch_size=32, shuffle=False, num_workers=0)
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
            pos_weight = make_pos_weight(tr_idx)
            best_auc, best_state, wait = -np.inf, None, 0
            max_epochs = 4 if FAST_MODE else 12
            for epoch in range(1, max_epochs + 1):
                model.train()
                for xb, yb, mb in train_loader:
                    xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
                    opt.zero_grad()
                    loss = masked_bce_loss(model(xb), yb, mb, pos_weight)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    opt.step()
                model.eval()
                probs = []
                with torch.no_grad():
                    for xb, yb, mb in valid_loader:
                        probs.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
                val_prob = np.vstack(probs)
                aucs = []
                for j in range(len(TARGETS)):
                    yv = Y[va_idx, j]
                    m = ~np.isnan(yv)
                    if m.sum() and len(np.unique(yv[m])) == 2:
                        aucs.append(safe_auc(yv[m], val_prob[m, j]))
                mean_auc = np.nanmean(aucs)
                print(f"   Epoch {epoch:03d} | val mean AUC={mean_auc:.4f}")
                if mean_auc > best_auc:
                    best_auc, best_state, wait = mean_auc, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
                else:
                    wait += 1
                if wait >= 4:
                    break
            if best_state:
                model.load_state_dict(best_state)
            torch.save({"model": model.state_dict(), "best_auc": best_auc}, ckpt)
        valid_loader = DataLoader(ImageToxDataset(va_idx), batch_size=32, shuffle=False, num_workers=0)
        probs = []
        model.eval()
        with torch.no_grad():
            for xb, yb, mb in valid_loader:
                probs.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
        oof[va_idx] = np.vstack(probs)
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    save_oof(name, oof)
    return oof

In [ ]:
if RUN_DEEP_MODELS and RUN_DENSENET121 and TORCHVISION_AVAILABLE:
    name = "densenet121"
    oof = load_oof(name)
    if oof is None:
        oof = train_densenet_oof(name)
    res_dense = summarize_oof_result("DL-5 DenseNet121", oof)
    res_dense.to_csv(RESULT_DIR / "metrics_densenet121.csv", index=False)
else:
    print("⏭️ DenseNet121 skipped.")

# Hybrid models

## Hybrid H-1 — D-MPNN-style + Fingerprint/Descriptors

In [ ]:
if RUN_DEEP_MODELS:
    name = "fusion_h1_graph_fp"
    oof = load_oof(name)
    if oof is None:
        epochs = 25 if FAST_MODE else 80
        oof = train_deep_oof(name, lambda: MultiTaskDNN(X_scaled_global.shape[1], len(TARGETS)),
                             max_epochs=epochs, batch_size=128)
    res_h1 = summarize_oof_result("H-1 Graph+FP Fusion", oof)
    res_h1.to_csv(RESULT_DIR / "metrics_h1_graph_fp.csv", index=False)
else:
    print("⏭️ H-1 skipped.")

## Hybrid H-2 — D-MPNN-style + SMILES Transformer probabilities

Prediction-level fusion for speed: average of graph branch and SMILES branch.

In [ ]:
name = "fusion_h2_graph_smiles"
oof = load_oof(name)
if oof is None:
    g = load_oof("dmpnn_fast")
    s = load_oof("smiles_transformer")
    if g is not None and s is not None:
        oof = np.nanmean(np.stack([g, s]), axis=0)
        save_oof(name, oof)
    else:
        print("⚠️ H-2 চালাতে dmpnn_fast এবং smiles_transformer OOF দরকার।")
if oof is not None:
    res_h2 = summarize_oof_result("H-2 Graph+SMILES Fusion", oof)
    res_h2.to_csv(RESULT_DIR / "metrics_h2_graph_smiles.csv", index=False)

## Hybrid H-3 — Proposed Fusion: Graph + SMILES + Fingerprint/Descriptors

Prediction-level ensemble of strongest three representation branches. This is faster than retraining a large 3-branch network.

In [ ]:
name = "fusion_h3_proposed"
oof = load_oof(name)
if oof is None:
    candidates = [load_oof("dmpnn_fast"), load_oof("smiles_transformer"), load_oof("mtl_dnn")]
    candidates = [c for c in candidates if c is not None]
    if len(candidates) >= 2:
        oof = np.nanmean(np.stack(candidates), axis=0)
        save_oof(name, oof)
    else:
        print("⚠️ H-3 চালাতে অন্তত দুইটি deep OOF prediction দরকার।")
if oof is not None:
    res_h3 = summarize_oof_result("H-3 Proposed Fusion", oof)
    res_h3.to_csv(RESULT_DIR / "metrics_h3_proposed.csv", index=False)

## Hybrid H-4 — Multitask CapsNet + RBF-SVM

CapsNet OOF probability + fingerprint feature space দিয়ে endpoint-wise RBF-SVM style hybrid।

In [ ]:
if RUN_HYBRID_SVM and RUN_SVM_RBF:
    name = "hybrid_capsnet_svm_rbf"
    oof = load_oof(name)
    if oof is None:
        caps_prob = load_oof("capsnet")
        if caps_prob is None:
            print("⚠️ CapsNet OOF পাওয়া যায়নি। আগে CapsNet cell run করো।")
        else:
            X_hybrid = np.hstack([X, np.nan_to_num(caps_prob, nan=0.5)]).astype(np.float32)
            X_backup = X.copy()
            X = X_hybrid
            def svm_factory(y):
                return SVC(C=4.0, gamma="scale", kernel="rbf", probability=True,
                           class_weight="balanced", cache_size=1200, random_state=SEED)
            oof = train_single_task_oof(name, svm_factory, scale=True)
            X = X_backup
    if oof is not None:
        res_caps_svm = summarize_oof_result("H-4 CapsNet + RBF-SVM", oof)
        res_caps_svm.to_csv(RESULT_DIR / "metrics_capsnet_svm_rbf.csv", index=False)
else:
    print("⏭️ CapsNet + RBF-SVM skipped.")

## Hybrid H-5 — DenseNet121 + RBF-SVM

DenseNet OOF probability + fingerprint feature space দিয়ে RBF-SVM hybrid।

In [ ]:
if RUN_HYBRID_SVM and RUN_SVM_RBF:
    name = "hybrid_densenet_svm_rbf"
    oof = load_oof(name)
    if oof is None:
        dense_prob = load_oof("densenet121")
        if dense_prob is None:
            print("⚠️ DenseNet121 OOF পাওয়া যায়নি। আগে DenseNet121 cell run করো।")
        else:
            X_hybrid = np.hstack([X, np.nan_to_num(dense_prob, nan=0.5)]).astype(np.float32)
            X_backup = X.copy()
            X = X_hybrid
            def svm_factory(y):
                return SVC(C=4.0, gamma="scale", kernel="rbf", probability=True,
                           class_weight="balanced", cache_size=1200, random_state=SEED)
            oof = train_single_task_oof(name, svm_factory, scale=True)
            X = X_backup
    if oof is not None:
        res_dense_svm = summarize_oof_result("H-5 DenseNet121 + RBF-SVM", oof)
        res_dense_svm.to_csv(RESULT_DIR / "metrics_densenet_svm_rbf.csv", index=False)
else:
    print("⏭️ DenseNet121 + RBF-SVM skipped.")

# Ensemble section

All available OOF predictions are collected. Endpoint-wise ensemble weight is learned only from OOF predictions.  
Final output remains ROC-AUC and Accuracy only.

In [ ]:
def collect_available_oofs():
    names = [
        "lightgbm", "xgboost", "random_forest", "extra_trees", "svm_rbf",
        "mtl_dnn", "dmpnn_fast", "smiles_transformer", "capsnet", "densenet121",
        "fusion_h1_graph_fp", "fusion_h2_graph_smiles", "fusion_h3_proposed",
        "hybrid_capsnet_svm_rbf", "hybrid_densenet_svm_rbf"
    ]
    found = {}
    for n in names:
        arr = load_oof(n)
        if arr is not None:
            found[n] = arr
    print("✅ Available OOF models:", list(found.keys()))
    return found

def endpoint_weighted_ensemble(oof_dict):
    model_names = list(oof_dict.keys())
    stacked = np.stack([oof_dict[n] for n in model_names], axis=0)
    ens = np.full_like(stacked[0], np.nan, dtype=np.float32)
    weight_table = []
    for j, t in enumerate(TARGETS):
        mask = ~np.isnan(Y[:, j])
        y = Y[mask, j].astype(int)
        aucs = []
        for i, n in enumerate(model_names):
            p = stacked[i, mask, j]
            if np.all(np.isnan(p)) or len(np.unique(y)) < 2:
                aucs.append(np.nan)
            else:
                aucs.append(safe_auc(y, p))
        aucs_arr = np.array(aucs, dtype=float)
        order = np.argsort(np.nan_to_num(aucs_arr, nan=-1))[::-1]
        top = [idx for idx in order[:5] if not np.isnan(aucs_arr[idx])]
        if not top:
            continue
        raw_w = np.maximum(aucs_arr[top] - 0.50, 1e-6)
        weights = raw_w / raw_w.sum()
        p = np.zeros(mask.sum(), dtype=float)
        for w, idx in zip(weights, top):
            p += w * stacked[idx, mask, j]
            weight_table.append({"Endpoint": t, "Model": model_names[idx], "Weight": w, "Model_AUC": aucs_arr[idx]})
        ens[mask, j] = p
    return ens, pd.DataFrame(weight_table)

if RUN_ENSEMBLE:
    oof_dict = collect_available_oofs()
    if len(oof_dict) >= 2:
        ensemble_oof, ens_weights = endpoint_weighted_ensemble(oof_dict)
        save_oof("final_weighted_ensemble", ensemble_oof)
        ens_weights.to_csv(RESULT_DIR / "ensemble_endpoint_weights.csv", index=False)
        res_ensemble = summarize_oof_result("✅ Final Weighted Ensemble", ensemble_oof)
        res_ensemble.to_csv(RESULT_DIR / "metrics_final_weighted_ensemble.csv", index=False)
        display(ens_weights.head(30))
    else:
        print("⚠️ Ensemble করতে অন্তত 2টি OOF model দরকার।")
else:
    print("⏭️ Ensemble skipped.")

# Final comparison summary

Only Mean ROC-AUC and Mean Accuracy will be compared.

In [ ]:
metric_files = sorted(RESULT_DIR.glob("metrics_*.csv"))
summary_rows = []
for p in metric_files:
    m = pd.read_csv(p)
    summary_rows.append({
        "Model": p.stem.replace("metrics_", ""),
        "Mean ROC-AUC": m["AUC-ROC"].mean(),
        "Mean Accuracy": m["Accuracy"].mean(),
        "Best Endpoint AUC": m.loc[m["AUC-ROC"].idxmax(), "Endpoint"],
        "Best AUC": m["AUC-ROC"].max(),
        "Best Endpoint Accuracy": m.loc[m["Accuracy"].idxmax(), "Endpoint"],
        "Best Accuracy": m["Accuracy"].max()
    })

summary = pd.DataFrame(summary_rows).sort_values(["Mean ROC-AUC", "Mean Accuracy"], ascending=False)
summary.to_csv(RESULT_DIR / "final_model_comparison_summary.csv", index=False)

print("✅ Final comparison complete.")
display(summary.style.format({
    "Mean ROC-AUC": "{:.6f}",
    "Mean Accuracy": "{:.6f}",
    "Best AUC": "{:.6f}",
    "Best Accuracy": "{:.6f}",
}))

In [ ]:
plt.figure(figsize=(10, max(4, 0.4 * len(summary))))
plot_df = summary.sort_values("Mean ROC-AUC", ascending=True)
plt.barh(plot_df["Model"], plot_df["Mean ROC-AUC"])
plt.xlabel("Mean ROC-AUC")
plt.title("Model comparison by Mean ROC-AUC")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, max(4, 0.4 * len(summary))))
plot_df = summary.sort_values("Mean Accuracy", ascending=True)
plt.barh(plot_df["Model"], plot_df["Mean Accuracy"])
plt.xlabel("Mean Accuracy")
plt.title("Model comparison by Mean Accuracy")
plt.tight_layout()
plt.show()

# Final note

`Mean ROC-AUC > 0.90` scientifically guarantee করা যায় না, কারণ score depends on split, missing labels, scaffold difficulty, and leakage control.  
এই notebook leakage-safe 5-fold mean result দেবে। Ensemble section available models থেকে best combined output আনবে।